# LiMP real evaluation

In [ ]:
import sys, pathlib, subprocess
ROOT = pathlib.Path.cwd()
assert (ROOT / "run.py").exists(), "Open this notebook from the extracted package folder"
def run(*args):
    subprocess.run([sys.executable, "run.py", *map(str, args)], check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import torch
print(torch.__version__, "CUDA:", torch.cuda.is_available())


## Main comparison

In [ ]:
DATA = "data/Claim_Credibility.xlsx"
OUT = "results_frozen"
SEEDS = ["42"]  # expand to ["42", "13", "21", "87", "100"] for run-stability SD
run("train", "--data", DATA, "--out", OUT, "--mode", "frozen", "--epochs", 77, "--seeds", *SEEDS)


In [ ]:
run("report", "--out", OUT, "--seed", 42, "--bootstrap", 2000)


##  retraining suites

In [ ]:
RUN_SUITES = False
if RUN_SUITES:
    for kind in ["component", "linguistic", "prototype", "hyperparameter"]:
        run("suite", "--data", DATA, "--out", OUT, "--kind", kind, "--seeds", *SEEDS)
    run("report", "--out", OUT, "--seed", 42)


##  SHAP/LIME and topics


In [ ]:
run("explain", "--out", OUT, "--model", "LiMP", "--seed", 42, "--samples", 3)
run("topics", "--out", OUT, "--topics", 6)


In [ ]:
import pandas as pd
display(pd.read_csv(pathlib.Path(OUT) / "report/table_main_model_performance.csv"))
print("Workbook:", pathlib.Path(OUT) / "report/All_Results_Tables.xlsx")


In [ ]:
from IPython.display import Image, display
for filename in ["combined_confusion_matrices.png", "combined_roc.png", "combined_calibration.png", "combined_recorded_compute_curves.png"]:
    display(Image(filename=str(pathlib.Path(OUT) / "report" / filename)))
